# Palmer Penguins — dfreport intro

Generates a self-contained offline HTML report from the Palmer Penguins dataset.

Demonstrates:
- Categorical dropdowns (species, island, sex) auto-detected from low-cardinality string columns
- Numeric min/max range filters (bill dimensions, flipper length, body mass)
- Scatter chart with hover detail
- Per-species summary stat table

**Dataset:** Palmer Penguins — Horst AM, Hill AP, Gorman KB (2020).  
License: CC0 (public domain). https://github.com/allisonhorst/palmerpenguins

In [ ]:
# --- path setup (local development) ---
# Remove these two lines once dfreport is installed via pip.
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../..').resolve()))

import pandas as pd
from dfreport import Report, ScatterChart, TOTAL_ROW_FLAG

In [ ]:
# Load dataset — CC0, sourced directly from the palmerpenguins GitHub repo
url = (
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/"
    "main/inst/extdata/penguins.csv"
)
df = pd.read_csv(url)

# Drop the 11 rows with any missing measurement
df = df.dropna().reset_index(drop=True)

print(f"{len(df)} penguins across {df['species'].nunique()} species, "
      f"{df['island'].nunique()} islands")
df.head()

In [ ]:
# Build per-species summary table
species_stats = (
    df.groupby('species', as_index=False)
    .agg(
        count=('bill_length_mm', 'count'),
        bill_length_mm=('bill_length_mm', 'mean'),
        bill_depth_mm=('bill_depth_mm', 'mean'),
        flipper_length_mm=('flipper_length_mm', 'mean'),
        body_mass_g=('body_mass_g', 'mean'),
    )
    .round(2)
)

# Append an ALL row (marked TOTAL_ROW_FLAG=True for bold styling)
total_row = pd.DataFrame([{
    'species':           'ALL',
    'count':             len(df),
    'bill_length_mm':    round(df['bill_length_mm'].mean(), 2),
    'bill_depth_mm':     round(df['bill_depth_mm'].mean(), 2),
    'flipper_length_mm': round(df['flipper_length_mm'].mean(), 2),
    'body_mass_g':       round(df['body_mass_g'].mean(), 2),
    TOTAL_ROW_FLAG:      True,
}])
species_stats = pd.concat([species_stats, total_row], ignore_index=True)

stat_col_defs = [
    {'key': 'species',           'label': 'Species',              'fmt': 'str'},
    {'key': 'count',             'label': 'Count',                'fmt': 'int'},
    {'key': 'bill_length_mm',    'label': 'Avg Bill Length (mm)', 'fmt': 'str'},
    {'key': 'bill_depth_mm',     'label': 'Avg Bill Depth (mm)',  'fmt': 'str'},
    {'key': 'flipper_length_mm', 'label': 'Avg Flipper (mm)',     'fmt': 'str'},
    {'key': 'body_mass_g',       'label': 'Avg Body Mass (g)',    'fmt': 'int'},
]

species_stats

In [ ]:
report_path = (
    Report(df, title='Palmer Penguins')
    # No date filter — the dataset has no date column.
    # Header bar gets species + island + sex dropdowns instead.
    .filters(['species', 'island', 'sex'])
    .chart(
        ScatterChart(
            x='bill_length_mm',
            y='bill_depth_mm',
            title='Bill Dimensions',
            x_label='Bill Length (mm)',
            y_label='Bill Depth (mm)',
            equal_axes=False,
            hover_columns=['species', 'island', 'sex', 'flipper_length_mm', 'body_mass_g'],
        )
    )
    # .summary_panel('bill_length_mm', 'bill_depth_mm', label='Bill stats')
    .stat_table(species_stats, stat_col_defs, title='Per-species summary')
    .table(
        categorical_threshold=10,
        # year is numeric but should be a categorical dropdown
        col_overrides={'year': {'is_categorical': True, 'is_numeric': False}},
    )
    .save('penguins_report.html')
)

print(f'Saved: {report_path.name}')

## What you should see

Open `penguins_report.html` in any browser.

- **Header bar** — three dropdowns: Species, Island, Sex.  
  Changing any one updates the scatter chart, stat panel, and data table simultaneously.
- **Bill Dimensions scatter** — each point is one penguin.  
  The average-X / average-Y crosshair moves as you filter.
- **Bill stats panel** — mean bill length, mean bill depth, std dev of the filtered set.
- **Per-species summary table** — pinned ALL row at the bottom; row counts reflect  
  the full unfiltered dataset (it's pre-aggregated, not live-filtered).
- **Data table** — paginates 50 rows at a time; each column gets an auto-detected  
  filter type (dropdown for species/island/sex, range for numerics).

### Notes on auto-detection

- `species`, `island`, `sex` — 3/3/2 unique values → categorical dropdowns ✓
- `year` — numeric dtype but overridden to categorical: renders as a dropdown showing 2007/2008/2009
- All `_mm` and `_g` columns — float dtype → min/max range inputs ✓